In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend
from scipy import signal


In [ ]:
dados = np.loadtxt('frs.txt')

In [ ]:

plt.figure(figsize=(15, 10))
plt.subplots_adjust(hspace=0.5)

plt.subplot(3,1,1)
plt.plot(dados[:,0],dados[:,1],color='blue')
plt.title('Médio-Lateral')
plt.xlabel('Tempo (s)')
plt.ylabel('Força (N)')

plt.subplot(3,1,2)
plt.plot(dados[:,0],dados[:,2],color='green')
plt.title('Ântero-posterior')
plt.xlabel('Tempo (s)')
plt.ylabel('Força (N)')

plt.subplot(3,1,3)
plt.plot(dados[:,0],dados[:,3],color='red')
plt.title('Vertical')
plt.xlabel('Tempo (s)')
plt.ylabel('Força (N)')

In [ ]:
### Esta rotina realizará o gráfico da análise espectral e residual além de calcular as frequências mediana, predominante e de 95% da força espectral.

# A rotina deverá ser invocada como:
# resultados = analiseespectral_residual(dados)
# Dados deverá ser o arquivo que possui uma série temporal.
# Criada por Vitor Cimonetti, em 03/05/2026

def analiseespectral_residual(dados):

    Freq_aquis = float(input('Qual a frequência de aquisição? '))
    if not np.isfinite(Freq_aquis) or Freq_aquis <= 0:
        raise ValueError('A frequência de aquisição deve ser positiva.')
    tem_tempo = input('Seus dados possuem coluna tempo? (s/n): ').strip().lower()

    if tem_tempo == 's':
        tempo = dados[:, 0]
        sinais = dados[:, 1:]
    else:
        tempo = np.arange(len(dados)) / Freq_aquis
        sinais = dados

    if len(sinais) <= 15 or sinais.shape[1] != 3 or not np.isfinite(sinais).all():
        raise ValueError('Informe três sinais com mais de 15 amostras finitas.')

    nomes = ['Força médio-lateral', 'Força ântero-posterior', 'Força vertical']

    resultados = {}

    for i in range(sinais.shape[1]):

        sinal = sinais[:, i]
        nome = nomes[i]

        # FFT
        Y = np.fft.fft(detrend(sinal, type='constant'), len(sinal))
        PY = np.abs(Y)**2 / len(sinal)

        f = Freq_aquis * np.arange(0, len(sinal)//2) / len(sinal)

        # Força espectral total
        forca_total = np.trapezoid(PY[0:len(f)], f)

        # Frequência mediana
        forca50 = forca_total * 0.5

        j = 2
        forca = 0
        while forca < forca50 and j <= len(f):
            forca = np.trapezoid(PY[0:j], f[0:j])
            j += 1

        freqMediana = f[j-2] if forca_total > 0 else np.nan

        # Frequência 95%
        forca95 = forca_total * 0.95

        j = 2
        forca = 0
        while forca < forca95 and j <= len(f):
            forca = np.trapezoid(PY[0:j], f[0:j])
            j += 1

        Freq95 = f[j-2] if forca_total > 0 else np.nan

        # Frequência predominante
        indice = np.argmax(PY[0:len(f)])
        freqPred = f[indice] if forca_total > 0 else np.nan

        # Gráficos
        fig, axs = plt.subplots(2, 1, figsize=(8, 8))

        axs[0].plot(tempo, sinal)
        axs[0].set_title(f' {nome}')
        axs[0].set_xlabel('Tempo (s)')
        axs[0].set_ylabel('Amplitude do sinal')

        axs[1].plot(f, PY[0:len(f)])
        axs[1].axvline(x=freqMediana, color='green', linewidth=3,
                       linestyle=':', label='Frequência mediana')
        axs[1].axvline(x=Freq95, color='red', linewidth=3,
                       linestyle=':', label='Frequência 95%')
        axs[1].axvline(x=freqPred, color='black', linewidth=2,
                       linestyle='--', label='Frequência predominante')

        axs[1].set_title(f'Espectro de potência - {nome}')
        axs[1].set_xlabel('Frequência (Hz)')
        axs[1].set_ylabel('Power')
        axs[1].legend()

        plt.tight_layout()
        plt.show()

# Análise residual

        residuo = []
        fc = []

        for freq_corte in np.arange(0.1, 20, 0.5):

            if freq_corte >= Freq_aquis / 2:
                continue

            b, a = signal.butter(4,freq_corte / (Freq_aquis / 2)) # calcula os coeficientes com filtro butter, passa-baixa, de 4a ordem

            sinal_filtrado = signal.filtfilt(b, a, sinal) # Filtra os dados

            residuo_freq = np.sqrt(np.mean((sinal_filtrado - sinal) ** 2)) # Calcula o residuo

            residuo.append(residuo_freq)
            fc.append(freq_corte)

        plt.figure(figsize=(8, 5))
        plt.plot(fc, residuo, marker='o')
        plt.xlabel('Frequência de corte (Hz)')
        plt.ylabel('Resíduo')
        plt.title(f'Análise residual - {nome}')
        plt.grid(True)
        plt.show()

        resultados[nome] = {
            'forca_total': forca_total,
            'freqMediana': freqMediana,
            'freqPred': freqPred,
            'Freq95': Freq95,

        }

    return resultados

In [ ]:
resultados = analiseespectral_residual(dados)

In [ ]:
Freq_aquis=1000 # Define a frequencia de aquisicao
b,a = signal.butter(4, 6/(Freq_aquis/2),'low')
serie_filtrada=signal.filtfilt(b,a,dados[:,3])

plt.figure(figsize=(10,5))
plt.plot(dados[:,0],dados[:,3],'r')
plt.plot(dados[:,0],serie_filtrada)
plt.xlabel('Tempo')
plt.ylabel('FRS (N)')
plt.title('Força Vertical')
plt.legend(['Sinal Bruto','Sinal filtrado'])
plt.show()